# Recurrent Neural Networks

# 0. Reminders

### 📸 Convolutional Neural Networks

In [ ]:
from keras import Sequential, Input
from tensorflow.keras import layers

model = Sequential()
model.add(Input(shape=(32,32,3)))
# Convolutional Layer + Max Pooling no. 1
model.add(layers.Conv2D(32, (5, 5),
                 padding='same',
                 strides = (1,1),
                 activation='relu'))
model.add(layers.MaxPooling2D(pool_size=(2, 2)))

# Convolutional Layer + Max Pooling no. 2
model.add(layers.Conv2D(32, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D(pool_size=(2, 2)))

# Convolutional Layer + Max Pooling no. 3
model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(layers.MaxPooling2D(pool_size=(2, 2)))

# Flattening Layer
model.add(layers.Flatten())

# Intermediate Dense Layer
model.add(layers.Dense(40, activation='relu'))

# Predictive Layer for a 10-class classification problem
model.add(layers.Dense(10, activation='softmax'))

### 🚀 Transfer Learning

- **Kernels extract spatial features from images.**
    - They are not specific to one task
    - Filters (= sets of kernels) can be re-used by loading an existing/pre-trained model
- The last layer of a CNN has to be set up according to the particular problem you want to solve.

```python
from tensorflow.keras.applications import vgg16
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers


# Loading the weights of a pre-trained ConvNet
pretrained = vgg16.VGG16(weights='imagenet', 
                         include_top=False,
                         input_shape=(256, 256, 3))

pretrained.trainable = False
# Optional, if you have time and want to re-train these weights, you can set it to True.

# Flatten + Intermediate Dense Layer + Predictive Layer
flatten_layer = layers.Flatten()
dense_layer = layers.Dense(100, activation='relu')
prediction_layer = layers.Dense(10, activation='softmax')

model = Sequential([
    pretrained, 
    flatten_layer, 
    dense_layer, 
    prediction_layer
])
```

# 1. Introduction to Recurrent Neural Networks

📈 Recurrent Neural Networks are Neural Networks specifically designed to deal with **sequences** as input data, i.e. **observations repeated throughout time**.

* 📸 Spatial features $\rightarrow$ Convolutional Neural Networks
* 📈 Temporal features $\rightarrow$ Recurrent Neural Networks

## 1.1 Examples

### Example 1: Prediction of future stock market values and trends

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/example_1.png" alt="Example 1" style="height:400px;"/>


### Example 2: Video prediction

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/example2.png" alt="Example 2" style="height:300px;"/>

🎥 Videos = sequences of images/frames

👉 Why not predicting the next image(s)?

### Example 3: Predicting the next word - Natural Language Processing

    
<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/rnn_text.png" alt="RNN for text" height=150 width=300>



🗣️ **Recurrent Networks are massively used for text!**

👉 Sentences = sequences of words

🤔 How do we turn a sequence of words into a sequence of mathematical inputs $x^t$? 

📅 See the next unit `Deep Learning - Natural Language Processing`

## 1.2. Let's talk about input shapes

### "Autoregressive" approach

During the Machine Learning lectures, you discovered Time Series as follows:

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/standard_time_series.png" alt="" style="height:300px;"/>

The algorithm would learn from the past values to predict the future values using temporal features such as the trend, the seasonality....

👉 _Example_: **ARIMA** models **recursively** predict the next data points **one after the other**

##### What follows is a slighly different approach...

###  General DL Framework

* <font color="blue"><b><i>Training:</i></b></font> Many pairs of ($X_i$, $y_i$) = (observation, target) are used to learn the patterns between the input and the output.
<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_1.png" alt="" style="height:250px;"/>


* <font color="magenta"><b><i>Prediction</i></b>:</font> New input $X_{new}$ $\rightarrow$ New prediction/output $y_{pred}$

❗️ Each individual observation $X_i$ can be of any type: scalar, vector, matrix (an image), ...

👉 `X.shape` = (<font color="blue">n_ROWS</font>, <font color="red">n_FEATURES</font>)

### <font color=red>RNN</font> specificity $\rightarrow$ input = sequences of observations

* <font color="blue"><b><i>Training</i></b>:</font> Each input $X_i$ consists of a **sequence of repeated observations** through time: 

$$ X_i = (X_i^{t=1}, X_i^{t=2}, X_i^{t=3}, ..., X_i^{t=m})$$


<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_2.png" alt="" style="height:200px;"/>

👉 `X.shape` = (<font color="blue">n_SEQUENCES</font>, <font color="#47bd1a">n_OBSERVATIONS</font>, <font color="red">n_FEATURES</font>) 

❗️ The input tensor $X$ has a new dimension, the <font color="#47bd1a"><b>temporal dimension</b></font> which corresponds to the <font color="#47bd1a"><b>number of observations through time</b>.</font>

### Example: 🌧️ Next day rainfall in different cities

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_3.png" alt="" style="height:300px;"/>

|         | <font color=blue># sequences</font> | <font color="#47bd1a"># temporal dimension</font>           | <font color=red># features     | shape      |
|---------|-------------|--------------------------------|----------------|------------|
| X.shape | <font color=blue>16 cities</font>   | <font color="#47bd1a">100 days of observation</font>        | <font color=red>1 (daily rain)</font>  | **(16,100,1)** |
| y.shape | <font color=blue>16 cities</font>   | <font color="#47bd1a">1 day to predict in the future</font> | <font color=red>1 (daily rain)</font>  | **(16,1,1)**   |

### It also works with multivariate observations

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_4.png" alt="" style="height:250px;"/>

Each observation can also be multivariate (e.g rain, temperature, atmospheric pressure)

|         | <font color=blue># sequences</font> | <font color="#47bd1a"># temporal dimension</font>           | <font color=red># features     | shape      |
|---------|-------------|--------------------------------|----------------|------------|
| X.shape | <font color=blue>16 cities</font>    | <font color="#47bd1a">100 days of observation</font>        | <font color=red>4 features</font>               | **(16,100,4)** |
| y.shape | <font color=blue>16 cities</font>    | <font color="#47bd1a">1 day to predict in the future</font> | <font color=red>1 (daily rain)</font>  | **(16,1,1)**   |

#### Comparing `X.shape` for different types of Neural Networks

| Input       | Type of neural network  | <font color=blue>Batch Size</font> | <font color="#47bd1a">Temporal Dimension</font>               | <font color=red>Features/Properties</font>          | X.shape              |
|-------------|-------------------------|------------|----------------------------------|------------------------------|--------------------|
| TS 📈 | Recurrent               | <font color=blue>16</font>         | <font color="#47bd1a">100 days of observation</font>          | <font color=red>4 atmospheric features</font>       | (<font color=blue>16</font>,<font color="#47bd1a">100</font>,<font color=red>4</font>)         |
| Video 📺       | Recurrent               | <font color=blue>16</font>         | <font color="#47bd1a">5 seconds x 60 frames per second</font> | <font color=red>(128, 128, 3) frames</font>         | (<font color=blue>16</font>,<font color="#47bd1a">300</font>,<font color=red>128,128,3</font>) |
| Image 📸       | Convolutional           | <font color=blue>16</font>         | -                                | <font color=red>(128, 128, 3) colored images</font> | (<font color=blue>16</font>,<font color=red>128,128,3</font>)     |
| Vector 🔢      | Dense | <font color=blue>16</font>         | -                                | <font color=red>7 features for example</font>       | (<font color=blue>16</font>,<font color=red>7</font>)             |

#### `X.shape` = (<font color="blue">n_SEQUENCES</font>, <font color="#47bd1a">n_OBSERVATIONS</font>, <font color="red">n_FEATURES</font>)

<hr>

***Mathematical notation***: $X_{\color{blue}{i},\color{#47bd1a}{j}}^{\color{red}{t}}$


 
- $\color{blue}{i}$ $\rightarrow$ <font color="blue"><b>sample/sequence</b></font> you are considering (<i>ex: <font color="blue">Paris</font>, <font color="blue">London</font>, <font color="blue">...</font></i>)
- $\color{red}{j}$ $\rightarrow$ <font color="red"><b>feature</b></font> that is measured (<i>ex: <font color="red">temperature</font>, <font color="red">humidity</font>, <font color="red">...</font></i>)
- $\color{#47bd1a}{t}$ $\rightarrow$ <font color="#47bd1a"><b>time</b></font> at which the **observation** is seen

#### ❌ Standard ML/DL algorithms are not able to handle this new temporal dimension

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/how_to.png" alt="" style="height:280px;"/>


### 🤔 What about the target `y`? 

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_5.png" alt="" style="height:300px;"/>

🔮 In the Time Series lecture, you were often predicting **the next observation**.

👉 Are we going to do the same with RNNs❓

#### 🔮 Prediction: The next observation

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_6.png" alt="" style="height:200px;"/>


👉 Predict `rain` on `day + 1`

* `X.shape` = (<font color="blue">None</font>, <font color="#47bd1a">100 days</font>, <font color="red">1 feature</font>)  
* `y.shape` = (<font color="blue">None</font>, <font color="#47bd1a">1 next day</font>, <font color="red">1 feature</font>)

    ℹ️ <font color="blue">None</font> is a placeholder for the <font color="blue">number of sequences</font> given, it is equal to <font color="blue">batch_size</font>

#### 🔮 Prediction: The next observation<u>S</u>

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_7.png" alt="" style="height:200px;"/>

👉 Predict `rain` on `[day + 1 ... day + 8]`

* `X.shape` = (<font color="blue">None</font>, <font color="#47bd1a">100 days</font>, <font color="red">1 feature</font>)
* `y.shape` = (<font color="blue">None</font>, <font color="#47bd1a">8 next days</font>, <font color="red">1 feature</font>)


#### 🔮 Prediction: The next observation<u>S</u>, but not necessarily right after the last seen value

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_8.png" alt="" style="height:200px;"/>

👉 Predict `rain` on `[day + 8 ... day + 16]`

* `X.shape` = (<font color="blue">None</font>, <font color="#47bd1a">100 days</font>, <font color="red">1 feature</font>)  
* `y.shape` = (<font color="blue">None</font>, <font color="#47bd1a">8 consecutive days in the future</font>, <font color="red">1 feature)</font>

#### 🔮 Prediction: The next observation<u>S</u>, but not necessarily consecutive

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_8_2.png" alt="" style="height:200px;"/>


👉 Predict `rain` for `5 specific days in the future`

* `X.shape` = (<font color="blue">None</font>, <font color="#47bd1a">100 days</font>, <font color="red">1 feature</font>)  
* `y.shape` = (<font color="blue">None</font>, <font color="#47bd1a">5 specific days in the future</font>, <font color="red">1 feature</font>)

#### 🔮 Prediction with multivariate inputs

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_9.png" alt="" style="height:200px;"/>
 

👉 Predict `rain` for the `8 next days` based on `multiple features [rain, humidity, temperature, pressure...]`

* `X.shape` = (<font color="blue">None</font>, <font color="#47bd1a">100 days of observation</font>, <font color="red">4 features</font>)  
* `y.shape` = (<font color="blue">None</font>, <font color="#47bd1a">8 next days</font>, <font color="red">1 feature = rain</font>)

#### 🔮Predicting multiple outputs

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_10.png" alt="" style="height:200px;"/>


👉 Predict `[rain, temperature]` for the `next 8 days` based on `past [rain, humidity, temperature, pressure...] features`

* `X.shape` = (<font color="blue">None</font>, <font color="#47bd1a">100 days of observation</font>, <font color="red">4 features</font>)  
* `y.shape` = (<font color="blue">None</font>, <font color="#47bd1a">8 next days</font>, <font color="red">2 features = [rain, temperature]</font>)

#### ❗️ The output is not necessarily similar to the past observations of X!

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_10.png" alt="" style="height:200px;"/>


👉 Predict `[avalanches, accidents]` for the `next 8 days` based on past `[rain, temperature, pressure...] features`

* `X.shape` = (<font color="blue">None</font>, <font color="#47bd1a">100 days of observation</font>, <font color="red">4 features = [rain,temperature,pressure]</font>)  
* `y.shape` = (<font color="blue">None</font>, <font color="#47bd1a">8 next days</font>, <font color="red">2 features = [avalanche, accidents]</font>)

#### 🔮 Prediction: using Temporal Data for Classification

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/exp_11.png" alt="" style="height:200px;"/>

👉 Predict the heartbeat category of a patient based on various  electrocardiograms (ECG)

* `X.shape` = (<font color="blue">None</font>, <font color="#47bd1a">100 heartbeats</font>, <font color="red">4 features</font>)  
* `y.shape` = (<font color="blue">None</font>, <font color="#47bd1a">1 heartbeat</font>, <font color="red">1 feature = at risk/healthy heart</font>)

### 🔮 Prediction Recap

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_description/comparison.png" alt="" style="height:200px;"/>

In Recurent Neural Networks:  
- **`Inputs X`** $\rightarrow$ **sequence** of **repeated observations** (univariate or multivariate)
    - <font color="blue">n_SEQUENCES</font> = number of sequences
    - <font color="#47bd1a">n_OBSERVATIONS</font> = INPUT_LENGTH = temporal dimension of the input
    - <font color="red">n_FEATURES</font> = features you are observing
    
✅ A dataset in RNN can have <font color="#47bd1a">different input lengths</font>

🗓️ We will discover at the end of the lecture how RNN can deal with varying lengths!

<hr>

📣 _Spoiler alert: we will use something called "padding" to deal with sequences that have different input lengths._

- **`Outputs y`** $\rightarrow$ is whatever you want to predict,
    - <font color="blue">n_SEQUENCES</font> = number of sequences
    - <font color="#47bd1a">OUTPUT_LENGTH</font> = how many timesteps you want to predict = temporal dimension for the output
    - <font color="red">n_FEATURES</font> = features you want to predict

👀 The feature(s) you want to predict is(are) not naturally similar to the features you are observing.

❌ The shape of what you want to predict must be the same for all the sequences. 

# 3. RNN architecture

### 3.1 Let's code! (Air Pollution Example)


👉 Let's assume that in <font color="blue">different cities</font>, you observed <font color="red">weather features</font> every day during <font color="#47bd1a">four days</font>:
- 🌡️ temperature
- 💨 wind speed
- ☁️ air pollution

You are trying to predict the amount of air pollution the **NEXT DAY**.

In [ ]:
import numpy as np

# --- SEQUENCE A (Paris)

day_1 = [10, 25, 50]  # OBSERVATION 1 [temperature, speed, pollution]
day_2 = [13, 10, 70]  # OBSERVATION 2 [temperature, speed, pollution]
day_3 = [ 9,  5, 90]  # OBSERVATION 3 [temperature, speed, pollution]
day_4 = [ 7,  0, 95]  # OBSERVATION 4 [temperature, speed, pollution]

sequence_a = [day_1, day_2, day_3, day_4]

y_a = [110] # Pollution at day 5

# --- SEQUENCE B (Berlin)
sequence_b = [[25, 20, 30], [26, 24, 50], [28, 20, 80], [22, 3, 110]]
y_b = [125]

# --- SEQUENCE C (London)
sequence_c = [[15, 10, 60], [25, 20, 65], [35, 10, 75], [36, 15, 70]]
y_c = [30]

X = np.array([sequence_a, sequence_b, sequence_c]).astype(np.float32)
y = np.expand_dims(np.array([y_a, y_b, y_c]).astype(np.float32), axis=-1)

print(X.shape)
print(y.shape)

(3, 4, 3)
(3, 1, 1)


❗️**Remember the shapes**❗️    

```python
X.shape # (n_SEQUENCES, n_OBSERVATIONS, n_FEATURES_to_observe)  
X.shape # (3 cities, 4 days, 3 features = [temperature, speed, pollution])

y.shape # (3 cities, 1 output per city, 1 feature to predict = [pollution])
```
<br>

In practice, you'll need many <font color="blue">more SEQUENCES</font> for your model to be able to <font color="blue">train</font> correctly! (>100)


Coding a RNN is quite straightforward with `tensorflow.keras` 👇

In [ ]:
# 0- Imports
from keras import Sequential, Input, layers
from tensorflow.keras.optimizers import Aadam

# 1- RNN Architecture
model = Sequential()
model.add(Input(shape=(4,3)))
model.add(layers.SimpleRNN(units=2, activation='tanh'))
model.add(layers.Dense(1, activation="linear"))

# 2- Compilation
model.compile(loss='mse',
              optimizer='rmsprop')  # Recommended optimizer for RNNs

# 3- Fit
model.fit(X, y,
          batch_size=16,
          epochs=10,
          verbose=0)

# 4- Predict
model.predict(X) # One prediction per city

array([[-0.69771504],
       [-0.6887897 ],
       [-0.11640026]], dtype=float32)

### 3.2 Recurrent Neural Networks under the hood

👇 Let's say we have a weather sequence for one *city*  $\color{red}{x} = \color{red}{[x^{(1)}, x^{(2)}, x^{(3)}, x^{(4)}]}$


```python

# ---- NEW CITY ----
x = [[10, 25, 50],  # x1
     [13, 10, 70],  # x2
     [ 9,  5, 90],  # x3
     [ 7,  0, 95]]  # x4
```


🎯 We want to predict $\color{blue}{y^{(5)}}$ = pollution at `day_5` using a RNN model.

``` python
y5 = ? # Pollution day 4+1 = 5

```

🔭 The RNN is fed <font color=red><u><b>one observation $\color{red}{x^{(k)}}$  at a time</b></u></font> (forward in time).

🧠 It maintains an <font color=green><u><b>internal state</b> $\color{green}{h}$ </u></font>that is updated at each time step.

**First time step ($t=1$)**  
<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/rnn_timestep_number_1.excalidraw.png" height=200 width=150>

- Initialize <font color=green>internal state</font> $\color{green}{h_0}$ 
- Compute <font color=green>new internal state</font> $\color{green}{h_1}$ using the <font color=green> initial internal state</font> $\color{green}{h_0}$ and  <font color=red> the obs.</font> $\color{red}{x^{(1)}}$ 
- Compute <font color=blue>layer output</font> $\color{blue}{y^{(1)}}$

**Second time step ($t=2$)**


<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/rnn_timestep_number_2.excalidraw.png" height=200 width=200>

- Compute <font color=green>new internal state</font> $\color{green}{h_2}$ using <font color=green>the previous internal state</font> $\color{green}{h_1}$ and  <font color=red> the obs.</font> $\color{red}{x^{(2)}}$ 
- Compute <font color=blue>layer output</font> $\color{blue}{y^{(2)}}$

**Last time step ($t=4$):**


<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/rnn_timestep_4.excalidraw.png" height=300 width=400>

Think about a Recurrent Network as a Neural Network that has a **memory** (<font color=green><b>the Internal State</b></font> $\color{green}{h}$) about past observations.

### 3.2 What are the trainable parameters of a recurrent layer?

In [ ]:
print("Input shape: ", X.shape)

model.summary()

Input shape:  (3, 4, 3)
Model: "sequential_6"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
simple_rnn_6 (SimpleRNN)     (None, 2)                 12        
_________________________________________________________________
dense_6 (Dense)              (None, 1)                 3         
Total params: 15
Trainable params: 15
Non-trainable params: 0
_________________________________________________________________


🤔 Why does this Recurrent layer have 12 trainable parameters?

**One single RNN layer** has a matrix $W$ of trainable params, such that

$$\color{green}{h^{(t)}} = f_W(\color{green}{h^{(t-1)}}, \color{red}{x^{(t)}})$$

- The **same** weights $W$ are used at every time step...
- ...to **recursively** compute its <font color=green>internal state</font> $\color{green}{h}$

<img src='https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/rnn_stanford.excalidraw.png'>

#### Let's zoom inside this function $f_W$ for one time step

<img src='https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/rnn_unit.png' width=470>  

* The <font color=red>current observation</font> $\color{red}{x^{(t)}}$ and the <font color=green>previous internal state</font> $\color{green}{h^{(t-1)}}$ are **concatenated** and fed into a **dense layer**...
* ...to output the <font color=green>new internal state</font> $\color{green}{h^{(t)}}$!

<hr>

We can _count_ the number of weights of the dense network above as $\color{green}{n_h}(\color{green}{n_h} + \color{red}{n_x} + 1)$.

- $\color{green}{n_h}$ = number of <font color=green>units</font> in our RNN layer
- $\color{red}{n_x}$ = number of <font color=red>features</font> measured per time-step (e.g. [temperature, humidity, ...])

🤓 _Mathematical formulation (optional)_

The weights matrix $W$ can be decomposed into $\color{red}{W_{xh}}$ and $\color{green}{W_{hh}}$ which are used to compute internal states:
<img src='https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/rnn_weights.png'>

The number of weights is, again, $\color{green}{n_h} * \color{red}{n_x} + \color{green}{n_h}*\color{green}{n_h} + (\color{green}{n_h} \text{ for the bias)}$ exactly as above ☝️ 

✏️ Let's compute the number of weights for our RNN example:

In [ ]:
X.shape

(3, 4, 3)

In [ ]:
n_x = 3 # 3 features per time-step (Temp, Humidity, Rain)
n_h = 2 # 2 memory units in our architecture

# How many weights are in our rnn layer?
n_h*n_x + n_h*n_h + n_h

12

In [ ]:
# Here are the 12 weights of our RNN layer
[w.numpy().shape for w in model.layers[0].weights]

[(3, 2), (2, 2), (2,)]

❗️ <u>Important note</u> ❗️ 

The number of weights in a Recurrent Layer has nothing to do with the length of our sequences (4 days in our example).

#### RNN layer output?

👉 A RNN layer outputs its internal state at the last time step: $\color{blue}{y_n} = \color{green}{h_n}$

<center><img src='https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/RNN_under_the_hood_full_no_seq.png' width=600 height=400></center>

<hr>

❌  $\color{blue}{y^{(t)}}$ is **NOT** the prediction/ the target. 

✅ $\color{blue}{y^{(t)}}$ is a vector of size `RNN_units` ($\color{green}{n_h = 2}$ in our example)...

✅ ... used as an input to the Dense layer to compute the <font color=blue>rain</font> at time $\color{blue}{t+1}$

<hr>
What does it mean intuitively❓

## Impact of the number of RNN units $\color{green}{h}$ ?

In [ ]:
# Consider this model with 10 RNN units
model_10 = Sequential()
model_10.add(layers.SimpleRNN(units=10))
model_10.add(layers.Dense(1, activation="linear"))

model_10.compile(loss='mse', optimizer='rmsprop')
model_10.fit(X, y, batch_size=16, epochs=10, verbose=0)
model_10.summary()

Model: "sequential_7"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
simple_rnn_7 (SimpleRNN)     (None, 10)                140       
_________________________________________________________________
dense_7 (Dense)              (None, 1)                 11        
Total params: 151
Trainable params: 151
Non-trainable params: 0
_________________________________________________________________


**☝️ Intuition:** With <font color=green> **10** </font> RNN units, this model:

- will try to _capture_ <font color=green> **10** </font> interesting _temporal features_ from the time-series 
    - (maybe: _mean_, _rate of increase_, complex auto-regressive feature, etc...who knows!)
- and combine them into 1 value for our regression task

💡 The number of units can be seen as the **number of memories about features maintained in parallel**.

# 4. Predicting an entire _sequence_

<img src='https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/sequence.png' width=500>

For each sequence/city:  

$X^{(t)}$ = weather observed at day $t$  
$y^{(t)}$ = Flu cases reported at day $t$

<hr>

Why not use $y$ at previous time steps as a feature?   

<hr>

🤷🏻‍♂️  You don't always have access to the previous values of $y$.

_Example: It may take one week to know the results of flu tests!_

`return_sequences=True` outputs $y^{(t)}$ at each time step.

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/rnn_return_sequences_true.excalidraw.png">

In [ ]:
model_2 = Sequential()
model_2.add(layers.SimpleRNN(units=2, return_sequences=True, activation='tanh'))
model_2.add(layers.Dense(1, activation='relu'))

#### ❗️ To predict a sequence,   `y_train` need to be sequences too ❗️

In [ ]:
import numpy as np

# --- SEQUENCE A (Paris)

sequence_a = [[10, 25, 50],  # OBS day 1
              [13, 10, 70],  # OBS day 2
              [ 9,  5, 90],  # OBS day 3
              [ 7,  0, 95]]  # OBS day 4

y_a = [70,   # flu cases day 1
       90,   # flu cases day 2
       95,   # flu cases day 3
       110,] # flu cases day 4

# --- SEQUENCE B (Berlin)
sequence_b = [[25, 20, 30], [26, 24, 50], [28, 20, 80], [22, 3, 110]]
y_b = [50, 80, 110, 125]

# --- SEQUENCE C (London)
sequence_c = [[15, 10, 60], [25, 20, 65], [35, 10, 75], [36, 15, 70]]
y_c = [65, 75, 70, 30]

X = np.array([sequence_a, sequence_b, sequence_c]).astype(np.float32)
y = np.expand_dims(np.array([y_a, y_b, y_c]).astype(np.float32),axis=-1)

print(X.shape)
print(y.shape)

(3, 4, 3)
(3, 4, 1)


❗️**Remember the wording**❗️    

`X.shape = (n_SEQUENCES, n_OBSERVATIONS, n_FEATURES)`  
`X.shape = (3 cities, 4 days, 3 features)`  
`y.shape = (3 cities, 4 days, 1 feature predicted each day)`  


In [ ]:
model_2 = Sequential()

model_2.add(layers.SimpleRNN(2, return_sequences=True))
model_2.add(layers.Dense(1, activation='relu'))

model_2.compile(loss='mse', optimizer='rmsprop')
model_2.fit(X, y, verbose=0)

In [ ]:
print("y_pred shape:", model_2.predict(X).shape)

y_pred shape: (3, 4, 1)


# 5. Stacking RNNs

### RNN Representation

If you start looking at different materials out there, be aware that they are usually represented in this way: 
    
<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/RNN_folded.png" style="height:300px;"/>

📈 $\color{green}{h}$ Units in Recurrent Networks capture $\color{green}{h}$ _temporal features_ from the Time Series

📸 Similarly, $\color{purple}{N}$ Filters in Convolutional Networks  would capture $\color{purple}{N}$ spatial features.

💡 We can create **more complex (second-order) features** by stacking another RNN on top!

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/RNN_stack.png" style="height:350px;"/>

👉 If you want to stack two RNN layers together, **DO NOT FORGET TO SET THE `return_sequences` ARGUMENT TO `True` on the first layer**!

❌ We usually don't see RNN models deeper than 3 or 4 layers.

In [4]:
model_stacked = Sequential()
model_stacked.add(Input(shape=(4,3)))
model_stacked.add(layers.SimpleRNN(10, return_sequences=True))
model_stacked.add(layers.SimpleRNN(3, return_sequences=False))
model_stacked.add(layers.Dense(1, activation='relu'))

model_stacked.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 4, 10)          │           140 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 3)              │            42 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186 (744.00 B)

 Trainable params: 186 (744.00 B)

 Non-trainable params: 0 (0.00 B)

# 6. RNN  Zoology

### RNNs suffer from vanishing gradient through time

<img src='https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/gradient_vanishing.png' width=700>

❗️Backpropagation through time❗️ Within a single layer RNN model.

- During backpropagation, the gradient vanishes to 0 as the time step decreases.
- As a result, simple RNNs are said to have __short memory__.

📚 [Towards Data Science - Michael Phi - Illustrated Guide to Recurrent Neural Networks](https://towardsdatascience.com/illustrated-guide-to-recurrent-neural-networks-79e5eb8049c9) 

### A family of different RNN architectures have been developed to correct this:

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/comparison.png" alt="Architectures comparison" />

🐣  **RNN (`SimpleRNN` in Keras)**: _introduced first, not always efficient_

💪 **LSTM  (Long Short Term Memory)**: _introduced to prevent the "vanishing gradient"_

🧨 **GRU (Gated Recurrent Units)**:  
- _introduced to reduce the number of parameters compared with LSTMs_
- _it implies a ***faster*** training, with potentially ***less training data***_

👇 Their syntax is very similar:

In [ ]:
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU

# Simple RNN
# -------------------------------------------------
model = Sequential()
model.add(SimpleRNN(units=10, activation='tanh'))

# LSTM
# -------------------------------------------------
model = Sequential()
model.add(LSTM(units=10, activation='tanh'))

# GRU
# -------------------------------------------------
model = Sequential()
model.add(GRU(units=10, activation='tanh'))

# Compile with 'rmsprop' rather than 'adam' (recommended)
# -------------------------------------------------
model.compile(loss='mse',
              optimizer='rmsprop')

💡 `activation='tanh'` is the default parameter. 

💪 This function squeezes numbers between -1 and 1.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/8/87/Hyperbolic_Tangent.svg/2560px-Hyperbolic_Tangent.svg.png" width=300 heigth=300>

💪 The second-order derivative of _tanh_ can sustain for a long range before going to zero, which is very helpful against the vanishing gradient problem.


<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/detanh2.png" width=300 heigth=300>

# 7. Variations

To conclude, there are many possible tasks that a Recurrent Neural Network can help with. The following terms are good to know even though it can be more complex in real life.

<hr>

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/many_to_many.png" alt="Many to Many" style="height:350px;"/>

<hr>

*Can you think about examples for each application?* (👉 [Answer](https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-recurrent-neural-networks))

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/many_to_many2.png">

# 8. Appendix (1): How to deal with input sequences of different lengths? 🤔

### Example

In [ ]:
import numpy as np

# --- SEQUENCE 1 (Paris) ---

sequence_1 = [[10, 25, 50],  # OBS day 1
              [13, 10, 70],  # OBS day 2
              [ 9,  5, 90],  # OBS day 3
              [ 7,  0, 95]]  # OBS day 4

y_1 = 110 # pollution day 5

# --- SEQUENCE 2 (Berlin) ---
sequence_2 = [[25, 20, 30],
              [26, 24, 50]]

y_2 = 125 # pollution day 3

# --- SEQUENCE 3 (London)
sequence_3 = [[15, 10, 60],
              [25, 20, 65],
              [35, 10, 75]]
y_3 = 30 # Pollution day 4

X = [sequence_1, sequence_2, sequence_3]
X = [np.array(_) for _ in X]
y = np.array([y_1, y_2, y_3]).astype(np.float32)

```python
model = Sequential()
model.add(SimpleRNN(1, activation='tanh')) 
model.add(Dense(1, activation="relu"))

# The compilation
model.compile(loss='mse', optimizer='rmsprop')

# The fit
model.fit(X, y, batch_size=16, epochs=10)
```
returns: 
``` shell
 ValueError: Data cardinality is ambiguous:
  x sizes: 4, 2, 3
  y sizes: 3
Make sure all arrays contain the same number of samples
```

### Problem!

This error is perfectly normal. Our sequences have different lengths:

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_pad.png" style="height:180px;"/>

However, the neural network must input a tensor of shape (<font color="blue">batch_size</font>, <font color="green">length of sequences</font>, <font color="red">features</font>). 


And with <font color="green">different lengths</font>, this tensor cannot be created yet.

While using data of different lengths is _legit_, it is not adapted to Tensors which are the mathematical objects we work with. 

## 8.1 Padding

Let's solve this problem with **padding**.

For **computational reasons**, you have to **pad** your input data with some values that will not be considered by the neural network, like so: 



<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_unpad.png"  style="height:180px;"/>

And there is a Keras function that serves this purpose!

📚 [tensorflow.keras.preprocessing.sequence.pad_sequences](https://www.tensorflow.org/api_docs/python/tf/keras/utils/pad_sequences)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_pad = pad_sequences(X, dtype='float32') # int32 by default
X_pad

array([[[ 10.,  25.,  50.],
        [ 13.,  10.,  70.],
        [  9.,   5.,  90.],
        [  7.,   0.,  95.]],

       [[ 25.,  20.,  30.],
        [ 26.,  24.,  50.],
        [ 28.,  20.,  80.],
        [ 22.,   3., 110.]],

       [[ 15.,  10.,  60.],
        [ 25.,  20.,  65.],
        [ 35.,  10.,  75.],
        [ 36.,  15.,  70.]]], dtype=float32)

#### ❗️ Avoid padding with existing values

- In the initial dataset, there are real 0's, and we can't differentiate them from the 0's used for padding.

- For that reason, pad with values that are not in the initial dataset, such as `-1000` here

<hr>

#### 💡 Pad at the end instead of the beginning `padding='post'` 

- Padding at beginning may influence internal RNN states!
- Padding at the end is safe because you will stop your prediction at the last "real" value anyway

In [ ]:
X_pad = pad_sequences(X, dtype='float32', padding='post', value=-1000)
X_pad

array([[[ 10.,  25.,  50.],
        [ 13.,  10.,  70.],
        [  9.,   5.,  90.],
        [  7.,   0.,  95.]],

       [[ 25.,  20.,  30.],
        [ 26.,  24.,  50.],
        [ 28.,  20.,  80.],
        [ 22.,   3., 110.]],

       [[ 15.,  10.,  60.],
        [ 25.,  20.,  65.],
        [ 35.,  10.,  75.],
        [ 36.,  15.,  70.]]], dtype=float32)

## 8.2 Masking Layer

ℹ️ Now, we have to tell our Neural Network to ignore these values on the forward/backward pass.

👩🏻‍💻 To do that, start your neural network with a [`Masking`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Masking) layer whose `mask_value` argument corresponds to the value you want to discard.


```python
from tensorflow.keras.layers import Masking
model.add(Masking(mask_value=-1000))
```


In [ ]:
from tensorflow.keras.layers import Masking

# –– Data
X = pad_sequences(X, dtype='float32', padding='post', value=-1000)
# X.shape == (3,4,3)

# –– Model
model = Sequential()
model.add(layers.Masking(mask_value=-1000, input_shape=(4,3)))
model.add(layers.SimpleRNN(units=2, activation='tanh'))
model.add(layers.Dense(10, activation='relu'))
model.add(layers.Dense(1, activation='linear'))

# –– Compilation
model.compile(loss='mse',
              optimizer='rmsprop') # Use `rmsprop`

# –– Fit
model.fit(X, y);

1/1 [==============================] - 1s 651ms/step - loss: 7229.3823


# 9. Appendix (2) - Single Time-Series? 🤔

😰 _What if we only had weather data about Paris_ ❓❗️

🧐 Can you think about a way to train Recurrent Neural Networks with only one 1-Dimensional Time Series?  
- What are our features $X$? 
- Our target $y$?

### Split your time series up into different <font color="#884dff">SEQUENCES</font> of (observations, target)

<center><img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/deep-learning/04/data_creation.png" style="height:300px;"/></center>

> If you have only one single 1D Time Series, you can extract samples of **`(sequence of observation, target)`** pairs from it and use these pairs to train a Recurrent Network.

❗️**Beware of data-leakage**❗️

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/time_series_cross_validation.png" alt="Time Series Cross Validation" width="800" height="400">

# Conclusion

📈 **Recurrent** Neural Networks are designed to deal with data that have a **temporal structure**.

- They can be used for different temporal tasks

- They depend on a <font color=green>hidden state</font>, which differs depending on the type of RNN (SimpleRNN, LSTM, GRU, ...)

- You can stack Recurrent Layers

- Don't forget to pad if you have sequences with different lengths

💡 <b><u>Pro Tips</u></b>:

✅ (1) Use `tanh` as the activation function of a Recurrent layer.

✅ (2) Use `rmsprop` as your optimizer.

# Bibliography

* 📚 [Stanford Cheat Sheet](https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-recurrent-neural-networks)  
* 📚 [Medium - Illustrative Guide to RNN](https://towardsdatascience.com/illustrated-guide-to-recurrent-neural-networks-79e5eb8049c9)  
* 📚 [Medium - Illustrative Guide to LSTM and GRUs](https://towardsdatascience.com/illustrated-guide-to-lstms-and-gru-s-a-step-by-step-explanation-44e9eb85bf21)  
* 📺 [RNN explained with 3*3 matrices - 21min](https://www.youtube.com/watch?v=UNmqTiOnRfg)  
* 📺 [_RNN & LSTM_ explained without math - 24min](https://www.youtube.com/watch?v=WCUNPb-5EYI)  
* 📺 [Stanford CS 231 Lecture on RNN - 1h20](https://www.youtube.com/watch?v=6niqTuYFZLQ)